Very Important Cell used to know which models are available

In [19]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
for m in client.models.list():
    if "gemini" in m.name and "embedding" not in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemini-4-hydrogen
models/gemini-4-neon


In [1]:
!pip -q install -U langchain langchain-community langchain-text-splitters langchain-chroma langchain-google-genai chromadb
!wget -q https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles.zip
!unzip -q new_articles.zip -d new_articles

y
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [ ]:
import os
import shutil

# Set your Gemini API Key (get one from Google AI Studio)
os.environ["GOOGLE_API_KEY"] = ""

In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Step 1: Load documents
loader = DirectoryLoader(
    "/content/new_articles/",
    glob="./*.txt",
    loader_cls=TextLoader,
)
documents = loader.load()

# Step 2: Chunking / Text Splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
texts = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(texts)}")

/tmp/ipykernel_447/1145975547.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Total chunks created: 233


In [7]:
import time
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

persist_directory = "db"

# Step 3: Initialize Gemini Embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

# Function to add documents with exponential backoff retry logic
def add_docs_with_retry(vector_store, docs, max_retries=5):
    delay = 15  # Start with a conservative 15-second delay for retry
    for attempt in range(max_retries):
        try:
            vector_store.add_documents(docs)
            return
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print(f"Rate limit hit. Waiting {delay} seconds before retrying (Attempt {attempt+1}/{max_retries})...")
                time.sleep(delay)
                delay *= 2  # Exponential backoff
            else:
                raise e
    raise RuntimeError("Failed to add documents due to repeated rate limit failures.")

# Step 4: Create an empty Chroma instance first
print("Initializing empty Chroma DB...")
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings,
)

# Ingest all documents in small batches of 20
batch_size = 20
print(f"Ingesting {len(texts)} chunks in batches of {batch_size}...")

for i in range(0, len(texts), batch_size):
    batch_docs = texts[i : i + batch_size]
    print(f"Processing batch starting at index {i}...")

    # Proactively sleep 10 seconds before every batch to prevent hitting limits
    if i > 0:
        time.sleep(10)

    add_docs_with_retry(vectordb, batch_docs)

print("Vector DB ingestion complete!")

Initializing empty Chroma DB...
Ingesting 233 chunks in batches of 20...
Processing batch starting at index 0...
Processing batch starting at index 20...
Processing batch starting at index 40...
Processing batch starting at index 60...
Processing batch starting at index 80...
Processing batch starting at index 100...
Processing batch starting at index 120...
Processing batch starting at index 140...
Processing batch starting at index 160...
Processing batch starting at index 180...
Processing batch starting at index 200...
Processing batch starting at index 220...
Vector DB ingestion complete!


In [10]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# Step 5: Configure Retriever
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

# Step 6: Initialize Gemini Chat Model & RAG Chain
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # Automatically tracks the latest stable Flash model
    temperature=0.2,
)

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Keep the answer concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Combine documents into the prompt, then wrap with the retriever
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [13]:
import time

def ask_question(query: str, max_retries: int = 3):
    for attempt in range(max_retries):
        try:
            response = rag_chain.invoke({"input": query})
            print(f"Answer:\n{response['answer']}\n")
            print("Sources:")
            for doc in response["context"]:
                print(f"- {doc.metadata.get('source', 'Unknown')}")
            print("-" * 50)
            return
        except Exception as e:
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                print(f"Model busy (503). Retrying in 10s... ({attempt + 1}/{max_retries})")
                time.sleep(10)
            else:
                raise

In [21]:
# Re-initialize the LLM using the fully qualified stable model name
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

llm = ChatGoogleGenerativeAI(
    model="models/gemini-3.5-flash-lite",
    temperature=0.2,
)

# Re-build the RAG chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# Run the queries
ask_question("How much money did Microsoft raise?")
ask_question("What is the news about Pando?")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:4109: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer:
Based on the provided context, Microsoft did not raise money; rather, Microsoft made a big investment in OpenAI, which is believed to be around $10 billion.

Sources:
- /content/new_articles/05-03-chatgpt-everything-you-need-to-know-about-the-ai-powered-chatbot.txt
- /content/new_articles/05-06-amazon-launches-free-channels-check-marks-come-to-gmail-and-openai-raises-more-moolah.txt
--------------------------------------------------
Answer:
Pando, a startup that develops fulfillment management technologies, announced that it has raised $30 million in a Series B round of funding, bringing its total capital raised to $45 million. The new funds will be used to expand the company's global sales, marketing, and delivery capabilities, augment its teams, and explore strategic partnerships and acquisitions.

Sources:
- /content/new_articles/05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt
- /content/new_articles/05-03-ai-powered-supply-chain-startup-pando-lands-30m-